# NeuroFace CV Pipeline

Interactive notebook for the NeuroFace Computer Vision pipeline:

1. **OVMS Health** — Query OpenVINO Model Server status
2. **Face Detection** — Run inference via OVMS REST API
3. **Label Registry** — List trained persons from LBPH recognizer
4. **Kafka Events** — Consume `cv.face.detections` topic
5. **Visualization** — Render bounding boxes on detected faces

In [ ]:
!pip install -q requests numpy opencv-python-headless matplotlib kafka-python Pillow

In [ ]:
import requests
import numpy as np
import json
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import time

OVMS_URL = "http://neuroface-ovms.neuroface.svc:8000"
NEUROFACE_URL = "http://neuroface-backend.neuroface.svc:8080"
KAFKA_BOOTSTRAP = "cdc-cluster-kafka-bootstrap.kafka-cdc.svc:9092"
KAFKA_TOPIC = "cv.face.detections"

print("Configuration loaded")
print(f"  OVMS:      {OVMS_URL}")
print(f"  NeuroFace: {NEUROFACE_URL}")
print(f"  Kafka:     {KAFKA_BOOTSTRAP}")

## 1. OVMS Health Check

Query `/v1/config` to verify the model server is running and `face-detection-retail-0005` is loaded.

In [ ]:
try:
    resp = requests.get(f"{OVMS_URL}/v1/config", timeout=10)
    config = resp.json()
    print("OVMS Status:", resp.status_code)
    print(json.dumps(config, indent=2))
    
    if "model_version_status" in str(config):
        print("\n>>> Model server is healthy and serving")
    else:
        print("\n>>> Model loaded but check status details above")
except requests.exceptions.ConnectionError:
    print("ERROR: Cannot reach OVMS. Is the neuroface-ovms deployment running?")

## 2. Face Detection Inference

Send an image to OVMS `face-detection-retail-0005` model and parse the detection results.

The model expects input `input.1` with shape `[1, 3, 300, 300]` (NCHW) and returns bounding boxes with confidence scores.

In [ ]:
def preprocess_image(image_data, target_size=(300, 300)):
    """Resize and normalize image for face-detection-retail-0005."""
    img = cv2.imdecode(np.frombuffer(image_data, np.uint8), cv2.IMREAD_COLOR)
    original = img.copy()
    resized = cv2.resize(img, target_size)
    blob = resized.transpose(2, 0, 1).astype(np.float32)  # HWC -> CHW
    blob = np.expand_dims(blob, axis=0)  # Add batch dim
    return blob, original


def run_inference(image_blob):
    """Send image to OVMS and return detections."""
    payload = {
        "inputs": [{
            "name": "input.1",
            "shape": list(image_blob.shape),
            "datatype": "FP32",
            "data": image_blob.flatten().tolist()
        }]
    }
    resp = requests.post(
        f"{OVMS_URL}/v2/models/face-detection-retail-0005/infer",
        json=payload,
        timeout=30
    )
    result = resp.json()
    if resp.status_code != 200:
        raise RuntimeError(f"OVMS returned {resp.status_code}: {result.get('error', result)}")
    return result


def parse_detections(result, original_shape, threshold=0.5):
    """Extract face bounding boxes above confidence threshold."""
    h, w = original_shape[:2]
    detections = []

    if "outputs" in result:
        output = result["outputs"][0]["data"]
    elif "output" in result:
        output = result["output"][0]["data"]
    else:
        print("DEBUG — OVMS response keys:", list(result.keys()))
        print("DEBUG — Full response (truncated):", json.dumps(result, indent=2)[:500])
        raise KeyError(
            f"Unexpected OVMS response format. Expected 'outputs' key, "
            f"got: {list(result.keys())}"
        )

    # Output format: [image_id, label, confidence, x_min, y_min, x_max, y_max]
    for i in range(0, len(output), 7):
        conf = output[i + 2]
        if conf > threshold:
            x1 = int(output[i + 3] * w)
            y1 = int(output[i + 4] * h)
            x2 = int(output[i + 5] * w)
            y2 = int(output[i + 6] * h)
            detections.append({
                "confidence": round(float(conf), 4),
                "bbox": {"x": x1, "y": y1, "w": x2 - x1, "h": y2 - y1}
            })
    return detections

print("Inference functions ready")

In [ ]:
# Download a test image with faces (Unsplash free-use photo)
TEST_IMAGE_URL = "https://avatars.githubusercontent.com/u/13861817?v=4"

try:
    img_resp = requests.get(TEST_IMAGE_URL, timeout=15)
    img_data = img_resp.content
    blob, original = preprocess_image(img_data)
    print(f"Image downloaded: {original.shape[1]}x{original.shape[0]}px")
    print(f"Input blob shape: {blob.shape}")
    
    # Run inference
    print("\nRunning face detection...")
    result = run_inference(blob)
    print(f"OVMS response keys: {list(result.keys())}")
    faces = parse_detections(result, original.shape)
    print(f"Detected {len(faces)} face(s)")
    
    for i, face in enumerate(faces):
        print(f"  Face {i+1}: confidence={face['confidence']}, bbox={face['bbox']}")

except requests.exceptions.ConnectionError:
    print("ERROR: Cannot reach OVMS or image URL")
    faces, original = [], None

## 3. NeuroFace Label Registry

Query the NeuroFace backend to list trained persons (LBPH recognizer labels).

In [ ]:
try:
    resp = requests.get(f"{NEUROFACE_URL}/api/labels", timeout=10)
    labels = resp.json()
    print(f"Trained persons: {len(labels.get('labels', []))}")
    print(json.dumps(labels, indent=2))
except requests.exceptions.ConnectionError:
    print("ERROR: NeuroFace backend not reachable")
    labels = {"labels": []}

## 4. Kafka Events — `cv.face.detections`

Consume recent events from the CV Kafka topic. Two event types flow through this topic:
- `person_registered` — emitted by the Labels Poller when a trained person is detected
- `ovms_model_status` — emitted by the OVMS Poller with model server health

In [ ]:
from kafka import KafkaConsumer
from kafka.errors import NoBrokersAvailable

events = []
try:
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=KAFKA_BOOTSTRAP,
        auto_offset_reset="earliest",
        consumer_timeout_ms=10000,
        value_deserializer=lambda m: json.loads(m.decode("utf-8"))
    )
    print(f"Connected to {KAFKA_BOOTSTRAP}")
    print(f"Consuming from '{KAFKA_TOPIC}' (10s window)...\n")
    
    for msg in consumer:
        event = msg.value
        events.append(event)
        evt_type = event.get("event_type", "unknown")
        ts = event.get("timestamp", "")
        
        if evt_type == "person_registered":
            print(f"[{ts}] PERSON: {event['person_name']} (faces: {event.get('face_count', '?')})")
        elif evt_type == "ovms_model_status":
            print(f"[{ts}] OVMS: {event.get('model_name', '?')} — server={event.get('model_server', '?')}")
        else:
            print(f"[{ts}] {evt_type}: {json.dumps(event)[:120]}")
    
    consumer.close()
    print(f"\nTotal events consumed: {len(events)}")

except NoBrokersAvailable:
    print("ERROR: Cannot connect to Kafka. Check SASL/SSL credentials if using secure connection.")
    print("Tip: For SASL_SSL, configure security_protocol and sasl_mechanism in KafkaConsumer.")

## 5. Visualization

Render detection results and event statistics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Face detection result
ax1 = axes[0]
if original is not None and len(faces) > 0:
    img_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    for face in faces:
        b = face["bbox"]
        cv2.rectangle(img_rgb, (b["x"], b["y"]), (b["x"]+b["w"], b["y"]+b["h"]), (0, 255, 0), 2)
        cv2.putText(img_rgb, f"{face['confidence']:.1%}", (b["x"], b["y"]-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    ax1.imshow(img_rgb)
    ax1.set_title(f"OVMS Face Detection — {len(faces)} face(s)")
else:
    ax1.text(0.5, 0.5, "No image / no detections", ha="center", va="center", fontsize=14)
    ax1.set_title("Face Detection")
ax1.axis("off")

# Right: Kafka event distribution
ax2 = axes[1]
if events:
    types = {}
    for e in events:
        t = e.get("event_type", "unknown")
        types[t] = types.get(t, 0) + 1
    colors = {"person_registered": "#2196F3", "ovms_model_status": "#FF9800"}
    bars = ax2.bar(types.keys(), types.values(),
                   color=[colors.get(k, "#999") for k in types.keys()])
    ax2.set_title(f"Kafka Events — {len(events)} total")
    ax2.set_ylabel("Count")
    for bar, count in zip(bars, types.values()):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(count), ha="center", fontweight="bold")
else:
    ax2.text(0.5, 0.5, "No Kafka events consumed", ha="center", va="center", fontsize=14)
    ax2.set_title("Kafka Events")

plt.tight_layout()
plt.show()

## Pipeline Summary

| Component | Endpoint | Status |
|-----------|----------|--------|
| OVMS | `neuroface-ovms.neuroface.svc:8000` | Check cell 1 |
| NeuroFace API | `neuroface-backend.neuroface.svc:8080` | Check cell 3 |
| Kafka Topic | `cv.face.detections` on `cdc-cluster` | Check cell 4 |
| Camel Notifier | `camel-cv-processor` in `kafka-cdc` | Consumes Kafka → Mailpit |